# Fine-Tuning Ministral 8B

 Google Colab Pro account with A100 and 40GB RAM.



## Installation

Clone the `mistral-finetune` repo:


In [ ]:
%cd /content/
!git clone https://github.com/mistralai/mistral-finetune.git

Install all required dependencies:

modify requirements.txt fire
simple-parsing
pyyaml
mistral-common>=1.3.1
safetensors
tensorboard
tqdm

torch==2.2
triton==2.2
xformers==0.0.24
numpy==1.26.4

In [ ]:
!pip uninstall -y numpy pandas pyarrow
!pip install --upgrade --force-reinstall numpy==1.26.4

!pip install fire simple-parsing pyyaml mistral-common==1.5.4 safetensors tensorboard tqdm torch==2.2 triton==2.2 xformers==0.0.24 numpy==1.26.4 pandas==2.2.2 pyarrow

## Model download

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

mistral_models_path = Path.home().joinpath('mistral_models', '8B-v0.3')
mistral_models_path.mkdir(parents=True, exist_ok=True)



In [ ]:
snapshot_download(repo_id="mistralai/Ministral-8B-Instruct-2410", allow_patterns=["params.json", "consolidated.safetensors", "tokenizer.model.v3"], local_dir=mistral_models_path)

! cp -r /root/mistral_models/8B-v0.3 /content/mistral_models
! rm -r /root/mistral_models/8B-v0.3

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

params.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

consolidated.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

In [ ]:
!ls /content/mistral_models

consolidated.safetensors  params.json


## Prepare dataset

To ensure effective training, mistral-finetune has strict requirements for how the training data has to be formatted. Check out the required data formatting [here](https://github.com/mistralai/mistral-finetune/tree/main?tab=readme-ov-file#prepare-dataset).



/content


In [ ]:
%cd /content/

# make a new directory called data
!mkdir -p data
# navigate to this data directory
%cd /content/data

# read data into a pandas dataframe
import numpy as np
import pandas as pd


df = pd.read_json('/content/data/function_callling_radare2_instruct.jsonl', lines=True)
# Mischia tutto il DataFrame in modo casuale e riproducibile
df_shuffled = df.sample(frac=1, random_state=200).reset_index(drop=True)
# Prendi i primi 100 per la valutazione
df_eval = df_shuffled.iloc[:100]
# Il resto per il training
df_train = df_shuffled.iloc[100:]
# Save to jsonl files
df_train.to_json("ultrachat_chunk_train.jsonl", orient="records", lines=True)
df_eval.to_json("ultrachat_chunk_eval.jsonl", orient="records", lines=True)

df_train


/content
/content/data


,messages,tools
100,"[{'role': 'system', 'content': ' ***RADARE2 MO...","[{'type': 'function', 'function': {'name': 'r2..."
101,"[{'role': 'system', 'content': ' ***RADARE2 MO...","[{'type': 'function', 'function': {'name': 'r2..."
102,"[{'role': 'system', 'content': ' ***RADARE2 MO...","[{'type': 'function', 'function': {'name': 'r2..."
103,"[{'role': 'system', 'content': ' ***RADARE2 MO...","[{'type': 'function', 'function': {'name': 'r2..."
104,"[{'role': 'system', 'content': ' ***RADARE2 MO...","[{'type': 'function', 'function': {'name': 'r2..."
...,...,...
3773,"[{'role': 'system', 'content': ' ***RADARE2 MO...","[{'type': 'function', 'function': {'name': 'r2..."
3774,"[{'role': 'system', 'content': ' ***RADARE2 MO...","[{'type': 'function', 'function': {'name': 'r2..."
3775,"[{'role': 'system', 'content': ' ***RADARE2 MO...","[{'type': 'function', 'function': {'name': 'r2..."
3776,"[{'role': 'system', 'content': ' ***RADARE2 MO...","[{'type': 'function', 'function': {'name': 'r2..."


In [ ]:
# save data into .jsonl files
df_train.to_json("ultrachat_chunk_train.jsonl", orient="records", lines=True)
df_eval.to_json("ultrachat_chunk_eval.jsonl", orient="records", lines=True)

In [ ]:
!ls /content/data
# navigate to the mistral-finetune directory
%cd /content/mistral-finetune/

corrected_radare2_instruct.jsonl  ultrachat_chunk_train.jsonl
ultrachat_chunk_eval.jsonl
/content/mistral-finetune


## Start training

In [ ]:
# these info is needed for training
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [ ]:
# define training configuration
# for your own use cases, you might want to change the data paths, model path, run_dir, and other hyperparameters

config = """
# data
data:
  instruct_data: "/content/data/ultrachat_chunk_train.jsonl"  # Fill
  data: ""  # Optionally fill with pretraining data
  eval_instruct_data: "/content/data/ultrachat_chunk_eval.jsonl"  # Optionally fill

# model
model_id_or_path: "/content/mistral_models"  # Change to downloaded path
lora:
  rank: 64

# optim
# tokens per training steps = batch_size x num_GPUs x seq_len
# we recommend sequence length of 32768
# If you run into memory error, you can try reduce the sequence length
seq_len: 5000
batch_size: 1
num_microbatches: 8
max_steps: 100
optim:
  lr: 1.e-4
  weight_decay: 0.1
  pct_start: 0.05

# other
seed: 0
log_freq: 1
eval_freq: 100
no_eval: False
ckpt_freq: 100

save_adapters: True  # save only trained LoRA adapters. Set to `False` to merge LoRA adapter into the base model and save full fine-tuned model

run_dir: "/content/test_ultra"  # Fill
"""

# save the same file locally into the example.yaml file
import yaml
with open('/content/example.yaml', 'w') as file:
    yaml.dump(yaml.safe_load(config), file)


In [ ]:
# make sure the run_dir has not been created before
# only run this when you ran torchrun previously and created the /content/test_ultra file
# ! rm -r /content/test_ultra

In [ ]:
# start training

#!torchrun --nproc-per-node=1 train --config /content/example.yaml
!torchrun --nproc-per-node 1 -m train /content/example.yaml

## Inference

In [ ]:
!pip install mistral_inference

In [ ]:
from mistral_inference.transformer import Transformer
from mistral_inference.generate import generate

from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
from mistral_common.protocol.instruct.messages import UserMessage
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.tokens.tokenizers.tekken import SpecialTokenPolicy

model = Transformer.from_folder("/content/mistral_models")  # change to extracted model dir
#tokenizer = MistralTokenizer.from_file("/content/mistral_models/tokenizer.model.v3")  # change to extracted tokenizer file
model.load_lora("/content/test_ultra/checkpoints/checkpoint_000100/consolidated/lora.safetensors")

tokenizer = MistralTokenizer.v3(is_tekken=True)  # change to extracted tokenizer file
model_name= "open-mistral-nemo"
tokenizer =MistralTokenizer.from_model(model_name)


/usr/local/lib/python3.11/dist-packages/mistral_common/tokens/tokenizers/mistral.py:152: FutureWarning: Calling `MistralTokenizer.from_model(..., strict=False)` is deprecated as it can lead to incorrect tokenizers. It is strongly recommended to use MistralTokenizer.from_model(..., strict=True)` which will become the default in `mistral_common=1.6.0`.If you are using `mistral_common` for open-sourced model weights, we recommend using `MistralTokenizer.from_file('<path/to/tokenizer/file>')` instead.
  warnings.warn(


In [ ]:

!huggingface-cli repo create mistral-r2-finetuned --type model

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    repo_id="e3m/mistral-r2-finetuned",  # Cambia con il tuo
    folder_path="/content/test_ultra/checkpoints/checkpoint_000100/consolidated",
    repo_type="model",
    commit_message="Upload LoRA fine-tuned weights"
)

In [ ]:
completion_request = ChatCompletionRequest(messages=[UserMessage(content=" What command writes base64 decoded data to address 0x08049000 in radare2?")])

tokens = tokenizer.encode_chat_completion(completion_request).tokens

out_tokens, _ = generate([tokens], model, max_tokens=64, temperature=0.0, eos_id=tokenizer.instruct_tokenizer.tokenizer.eos_id)

# Fix: cambia la policy per permettere la decodifica
tokenizer.instruct_tokenizer.tokenizer.special_token_policy = SpecialTokenPolicy.IGNORE

result = tokenizer.instruct_tokenizer.tokenizer.decode(out_tokens[0])

print(result)

[{"name": "r2cmd", "arguments": {"command": "w6e SGVsbG8= @ 0x08049000"}}]
